# Mobile Money Agent Network: Audit and Cleaning
Audit, clean, merge and summarise `agent_transactions.csv` with the agent register `agents.csv`. All cleaning rules live in `cleaning_pipeline.py`; this notebook calls them and interprets the results.
Regenerate the raw files first with `python generate_data.py`.

In [1]:
import pandas as pd
from cleaning_pipeline import audit_agent_data, clean_agent_data
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)

## 1. Load and audit the raw data
Raw files are read as text (`dtype=str`, `keep_default_na=False`) so that nothing is silently converted: the literal `N/A` and blanks stay visible and are handled by explicit rules.

In [2]:
txns_raw = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)
agents = pd.read_csv("agents.csv")
print("transactions:", txns_raw.shape, "| agents:", agents.shape)
txns_raw.head()

transactions: (612, 5) | agents: (20, 3)


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


In [3]:
# Cross-check: what plain pd.read_csv would do (N/A silently becomes NaN)
plain = pd.read_csv("agent_transactions.csv")
print("plain read, amount dtype:", plain["amount"].dtype, "| isna:", plain["amount"].isna().sum(),
      "| literal 'N/A' strings:", (txns_raw["amount"].str.strip() == "N/A").sum(),
      "| empty strings:", (txns_raw["amount"].str.strip() == "").sum())

plain read, amount dtype: float64 | isna: 22 | literal 'N/A' strings: 7 | empty strings: 15


In [4]:
audit = audit_agent_data(txns_raw, agents)
audit

,issue,dimension,column,rows_affected
0,Amount missing (blank/NaN) or literal 'N/A',Completeness,amount,22
1,Amount present but not numeric,Validity,amount,0
2,Amount zero or negative,Validity,amount,0
3,txn_date not parseable as a date,Validity,txn_date,0
4,Non-ISO date format (YYYY/MM/DD),Consistency,txn_date,5
5,txn_type has inconsistent case/whitespace,Consistency,txn_type,384
6,txn_type not mappable to Deposit/Withdrawal,Validity,txn_type,0
7,Fully identical duplicate rows (extra copies),Uniqueness,all columns,11
8,"Repeated txn_id (extra copies, incl. conflicting copies)",Uniqueness,txn_id,12
9,agent_id not in agent register,Validity,agent_id,8


Notes on the audit: `rows_affected` counts rows in the **raw** extract, so the 22 missing amounts include two rows that are duplicate copies and the 8 unknown-agent rows include one duplicate copy. The six issues injected into the extract are (1) missing/`N/A` amounts, (2) messy `txn_type`, (3) mixed date formats, (4) duplicates, (5) `250000` placeholder amounts and (6) unknown agent `AG099`. The IQR row also catches ordinary high values, discussed in section 6.

## 2. Run the cleaning pipeline

In [5]:
clean, log = clean_agent_data(txns_raw, agents)
log

,step,rule,rows_before,rows_after,rows_affected
0,1,"Trim whitespace in txn_id, agent_id, txn_type",612,612,0
1,2,Standardise txn_type to Deposit/Withdrawal (invalid -> NaN),612,612,384
2,3,"Convert amount to numeric (blank, 'N/A', text, <=0 -> NaN)",612,612,22
3,3,Parse txn_date to datetime (YYYY/MM/DD and YYYY-MM-DD),612,612,5
4,4,Drop duplicate txn_id (of which 11 were identical rows),612,600,12
5,5,Impute missing/invalid amount with in-range median by txn_type; flag amount_...,600,600,20
6,6,"Flag outliers outside IQR fences [-1,928, 6,084] (kept, not removed)",600,600,40
7,7,Left-join agent register on agent_id (unknown agents kept as 'Unknown'),600,600,7


## 3. Uniqueness

In [6]:
print("raw rows:", len(txns_raw), "| clean rows:", len(clean), "| removed:", len(txns_raw) - len(clean))
print("identical duplicate rows:", txns_raw.duplicated().sum(), "| repeated txn_id:", txns_raw["txn_id"].duplicated().sum())
dup_ids = txns_raw[txns_raw["txn_id"].duplicated(keep=False)].sort_values("txn_id")
dup_ids.groupby("txn_id").nunique().max()  # which columns differ between copies?

raw rows: 612 | clean rows: 600 | removed: 12
identical duplicate rows: 11 | repeated txn_id: 12


agent_id    2
txn_type    1
amount      1
txn_date    1
dtype: int64

**12 duplicate transactions removed** (612 to 600). Eleven are exact copies. The twelfth has the same `txn_id` but a corrupted `agent_id` (`AG099`) in one copy, so a full-row check alone would miss it. Rule: deduplicate on `txn_id`, keeping the copy whose agent exists in the register.

## 4. Consistency, validity and types

In [7]:
print(txns_raw["txn_type"].map(repr).value_counts(), "\n")
print(clean["txn_type"].value_counts(), "\n")
print(clean.dtypes)
print("date range:", clean["txn_date"].min().date(), "to", clean["txn_date"].max().date())

txn_type
'deposit'       142
'withdrawal'    131
'Deposit'       128
'DEPOSIT '      111
'Withdrawal'    100
Name: count, dtype: int64 

txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64 

txn_id                    string
agent_id                  string
county                       str
onboarded                    str
txn_type                     str
amount                   Float64
txn_date          datetime64[us]
amount_imputed              bool
is_outlier               boolean
unknown_agent               bool
dtype: object
date range: 2026-03-01 to 2026-04-29


`txn_type` had 5 spellings (case and trailing whitespace) and is now exactly `Deposit` / `Withdrawal`. Five dates used `YYYY/MM/DD`; all are parsed to real datetimes and the range is 1 Mar to 29 Apr 2026. `amount` is numeric (`N/A` and blanks became `NaN` before any filling).

## 5. Completeness: missing and invalid amounts

In [8]:
print("imputed:", clean["amount_imputed"].sum(), "of", len(clean), f"({clean['amount_imputed'].mean():.1%})")
clean.groupby("txn_type")["amount"].median().to_frame("median_used_or_observed")

imputed: 20 of 600 (3.3%)


,median_used_or_observed
txn_type,
Deposit,1755.0
Withdrawal,1716.0


**Decision:** 20 transactions (3.3%) have no usable amount after de-duplication. I keep the rows (they are real events and count towards transaction volumes), fill the amount with the **median of normal-range amounts for the same transaction type**, and set `amount_imputed = True`. The median is used because the amounts are right-skewed and the 250,000 values would distort a mean. A fabricated amount is not evidence of money moved, so **all value totals and averages in the summaries exclude imputed rows**; only counts include them. Dropping them would understate activity and bias agent comparisons; leaving `NaN` would break downstream arithmetic. The recommended real fix is to recover the amounts from the legacy system.

## 6. Outliers (IQR) and the KES 250,000 values

In [9]:
obs = clean.loc[~clean["amount_imputed"], "amount"].astype(float)
q1, q3 = obs.quantile([.25, .75]); iqr = q3 - q1
print(f"Q1={q1:,.0f} Q3={q3:,.0f} IQR={iqr:,.0f} fences=({q1-1.5*iqr:,.0f}, {q3+1.5*iqr:,.0f})")
print("flagged:", clean["is_outlier"].sum())
big = clean[clean["amount"] == 250000]
print("rows at exactly 250,000:", len(big), "| as multiple of median:", round(250000 / obs.median()))
print("largest other flagged value:", clean.loc[clean["is_outlier"] & (clean["amount"] != 250000), "amount"].max())
big[["txn_id", "agent_id", "county", "txn_type", "txn_date", "amount"]]

Q1=1,077 Q3=3,080 IQR=2,003 fences=(-1,928, 6,084)
flagged: 40
rows at exactly 250,000: 5 | as multiple of median: 141
largest other flagged value: 28847.0


,txn_id,agent_id,county,txn_type,txn_date,amount
306,TX00307,AG012,Kisumu,Deposit,2026-03-02,250000.0
315,TX00316,AG014,Nairobi,Deposit,2026-04-01,250000.0
424,TX00425,AG005,Mombasa,Withdrawal,2026-03-02,250000.0
463,TX00464,AG009,Nakuru,Withdrawal,2026-04-11,250000.0
594,TX00595,AG016,Kiambu,Deposit,2026-03-11,250000.0


The IQR rule flags 40 amounts, but they are two different things:
- **35 ordinary high values (6,158 to 28,847).** The data is log-normal (right-skewed), so a symmetric IQR fence over-flags the natural tail. These are plausible transactions and are kept.
- **5 values of exactly KES 250,000**, about 140x the median and 8.7x the next-largest amount. They are identical, round, spread over 5 different agents in 5 different counties and both transaction types. That pattern points to a **system default, test value or limit cap rather than genuine customer behaviour**.

**Recommendation:** do not delete and do not trust them. Keep them flagged (`is_outlier`), exclude them from average-value KPIs until verified, and refer the 5 `txn_id`s to the provider's operations/fraud team to check against the ledger and the agent float (a round maximum value is also a classic structuring/fraud signal). Totals below include them; the sensitivity check shows the effect.

In [10]:
valid = clean[~clean["amount_imputed"]]
pd.DataFrame({"total_value": [valid["amount"].sum(), valid.loc[valid["amount"] != 250000, "amount"].sum()],
              "avg_value": [valid["amount"].mean(), valid.loc[valid["amount"] != 250000, "amount"].mean()]},
             index=["with 250,000 rows", "without 250,000 rows"]).round(0)

,total_value,avg_value
"with 250,000 rows",2652897.0,4574.0
"without 250,000 rows",1402897.0,2440.0


## 7. Merge with the agent register

In [11]:
print("rows before merge (deduplicated):", len(txns_raw) - 12, "| rows after merge:", len(clean))
print("unknown-agent transactions:", clean["unknown_agent"].sum(), "| ids:", clean.loc[clean["unknown_agent"], "agent_id"].unique())
print("register agents with no transactions:", (~agents["agent_id"].isin(clean["agent_id"])).sum())
clean["county"].value_counts()

rows before merge (deduplicated): 600 | rows after merge: 600
unknown-agent transactions: 7 | ids: <StringArray>
['AG099']
Length: 1, dtype: string
register agents with no transactions: 0


county
Nairobi    180
Mombasa    133
Kisumu     111
Nakuru      99
Kiambu      70
Unknown      7
Name: count, dtype: int64

A **left join** (`validate="m:1"`) keeps every transaction, so the row count is unchanged (600 before and after). Seven transactions reference `AG099`, which is not in the register (8 in the raw extract; one was a duplicate copy). I keep them, label `county = "Unknown"` and set `unknown_agent = True` rather than dropping real transactions or guessing a county. They are shown as their own line in the summaries and should be referred to the agent-onboarding team.

## 8. Summaries

In [12]:
valid = clean[~clean["amount_imputed"]]
vol = clean.groupby(["county", "txn_type"]).size().rename("n_transactions")
val = valid.groupby(["county", "txn_type"])["amount"].agg(total_value="sum", avg_value="mean")
county_summary = pd.concat([val, vol], axis=1)[["total_value", "n_transactions", "avg_value"]].round(0)
county_summary

total_value  n_transactions  avg_value
county  txn_type                                          
Kiambu  Deposit        334737.0              43     7785.0
        Withdrawal      75274.0              27     2895.0
Kisumu  Deposit        431091.0              74     6072.0
        Withdrawal      74598.0              37     2072.0
Mombasa Deposit        141880.0              72     2086.0
        Withdrawal     368592.0              61     6042.0
Nairobi Deposit        536484.0             118     4706.0
        Withdrawal     154930.0              62     2671.0
Nakuru  Deposit        181337.0              60     3126.0
        Withdrawal     326818.0              39     8600.0
Unknown Deposit         25279.0               6     4213.0
        Withdrawal       1877.0               1     1877.0

In [13]:
clean_w = valid.assign(week_start=valid["txn_date"].dt.to_period("W-SUN").dt.start_time.dt.date)
weekly = clean_w.pivot_table(index="week_start", columns="county", values="amount",
                             aggfunc="sum", fill_value=0, margins=True, margins_name="Total").round(0)
weekly

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown,Total
week_start,,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,967.0,0.0,16665.0
2026-03-02,18317.0,287933.0,283893.0,68967.0,56574.0,0.0,715684.0
2026-03-09,265995.0,23186.0,28701.0,51084.0,28730.0,0.0,397696.0
2026-03-16,22054.0,24572.0,23368.0,41331.0,39795.0,1639.0,152759.0
2026-03-23,11504.0,27077.0,17050.0,39597.0,17222.0,0.0,112450.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0,411613.0
2026-04-06,15030.0,15341.0,45316.0,55010.0,276609.0,9824.0,417130.0
2026-04-13,12265.0,25863.0,26730.0,54656.0,24376.0,0.0,143890.0
2026-04-20,24507.0,37439.0,16687.0,61634.0,30862.0,1877.0,173006.0


The weekly spikes are almost entirely the five KES 250,000 rows (week of 2 Mar: Kisumu and Mombasa; 9 Mar: Kiambu; 30 Mar: Nairobi; 6 Apr: Nakuru). Without them the weekly pattern is flat, roughly KES 100,000 to 200,000 per week network-wide, so county rankings by value should be read with that caveat until the 250,000 rows are verified. By transaction count, Nairobi leads clearly (180 of 600) and deposits outnumber withdrawals 373 to 227.

In [14]:
top = county_summary["total_value"].groupby("county").sum().drop("Unknown", errors="ignore").sort_values(ascending=False)
print((top / top.sum()).round(3).to_string())

county
Nairobi    0.263
Mombasa    0.194
Nakuru     0.194
Kisumu     0.193
Kiambu     0.156


## 9. Export

In [15]:
import os
os.makedirs("outputs", exist_ok=True)
out = clean.copy(); out["txn_date"] = out["txn_date"].dt.strftime("%Y-%m-%d")
out.to_csv("outputs/agent_transactions_clean.csv", index=False)
chk = pd.read_csv("outputs/agent_transactions_clean.csv")
print(chk.shape, "| unique txn_id:", chk["txn_id"].is_unique, "|", sorted(chk["txn_type"].unique()))

(600, 10) | unique txn_id: True | ['Deposit', 'Withdrawal']


# Data Quality Report

**Scope:** 612 raw transactions (1 Mar to 29 Apr 2026) and a 20-agent register. **Result:** 600 clean transactions; 12 duplicates removed; nothing else deleted.

| Dimension | Finding | Treatment |
|---|---|---|
| **Completeness** | 22 raw amounts missing (15 blank, 5 text `N/A`, plus 2 duplicate copies); 20 remain after de-duplication (3.3%). | Median by transaction type, `amount_imputed` flag, excluded from value totals. |
| **Uniqueness** | 12 repeated `txn_id`s (11 exact copies, 1 copy with a corrupted agent). | Deduplicated on `txn_id`; the copy with a valid agent is kept. |
| **Consistency** | `txn_type` had 5 spellings (case, trailing space) across 384 rows; 5 dates used `YYYY/MM/DD`. | Standardised to `Deposit`/`Withdrawal` and ISO dates. |
| **Validity** | `AG099` (7 clean rows) is not in the register; no non-numeric amounts beyond `N/A`, no non-positive amounts, no unparseable dates. | Left join, `county = "Unknown"`, `unknown_agent` flag; rows kept. |
| **Accuracy** | 5 identical KES 250,000 amounts (about 140x the median); IQR flags 40 values in total, 35 of them ordinary right-skew. | Flagged with `is_outlier`, kept, to be verified against the ledger. Excluded from average KPIs until confirmed. |
| **Timeliness** | All transactions fall within a 60-day window and after each agent's onboarding date. No stale or future-dated records. | None needed. |

**Limitations:** imputed amounts are placeholders, county rankings are sensitive to the 250,000 rows, and 7 transactions cannot be attributed to a county.

**Privacy (Kenya Data Protection Act 2019).** If the extract also held customer phone numbers, the phone number (an MSISDN that identifies a natural person) would be personal data. Combined with transaction amounts, dates and agent location, it would reveal financial behaviour and movements, which makes the record sensitive in practice. Under the Act the provider must collect only what is necessary (data minimisation) and have a lawful basis; protect it with encryption at rest and in transit and role-based access; and **pseudonymise** the phone number (a salted hash or tokenised customer ID) before it reaches analysts, since this analysis needs only an ID, not the number. Retain it only as long as required, log access, notify the Data Protection Commissioner of a breach within 72 hours, and honour data-subject rights (access, correction, erasure). Agent identifiers are business data, but agent names or national IDs would be personal data too.